# Init

In [0]:
import pyspark.sql.functions as f
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

# read from bronze

In [0]:
df=spark.table("workspace.bronze.crm_cust_info")


#Data Transformation

##Trimming

In [0]:
for field in df.schema.fields:
  if field.dataType == StringType():
    df = df.withColumn(field.name, trim(col(field.name)))


##Normalization 

In [0]:
df = (
    df
    .withColumn(
        "cst_marital_status",
        f.when(f.upper(f.col("cst_marital_status")) == "S", "Single")
         .when(f.upper(f.col("cst_marital_status")) == "M", "Married")
         .otherwise("n/a")
    )
    .withColumn(
        "cst_gndr",
        f.when(f.upper(f.col("cst_gndr")) == "F", "Female")
         .when(f.upper(f.col("cst_gndr")) == "M", "Male")
         .otherwise("n/a")
    )
)


#Renaming columns 

In [0]:
RENAME_MAP = { #dictonary
    "cst_id": "customer_id",
    "cst_key": "customer_number",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)


#Remove Records with Missing Customer ID

In [0]:
df = df.filter(col("cst_id").isNotNull())

#Sanity checks of dataframe

In [0]:
df.limit(10).display()

#Write to silver layer

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_customers")